# Rich-response distillation, then retrain the joint planner

The plan corpus has correct but **terse** responses, so the planner learns to answer tersely.
Here a stronger teacher (Qwen2.5-7B, 4-bit) **expands each brief answer into a detailed one that
follows the plan** (keeping facts/numbers exact). Then we retrain the one-model joint planner on
these rich targets so planned answers are genuinely thorough.

**Runtime -> Change runtime type -> T4 GPU**, then run cells top to bottom.

In [ ]:
!git clone -b one-model-joint https://github.com/sinha-k-prat/latent_agentic_planning.git
%cd latent_agentic_planning
!pip -q install -r requirements.txt
!pip -q uninstall -y torchao

In [ ]:
import torch
assert torch.cuda.is_available(), 'No GPU — set Runtime > Change runtime type > T4 GPU'
print('GPU:', torch.cuda.get_device_name(0))

## 1) Distill rich responses (Qwen2.5-7B, 4-bit)
Downloads the 7B once (~5 GB) then rewrites **300** brief answers into detailed, plan-following
ones. Sized to finish comfortably in one session (~30-45 min). To do more, raise `--limit` here
and the train sizes in step 2 (keep `train+held+calib <= limit`). With an Anthropic key you can
swap to `--teacher api --model claude-opus-4-8` (faster + better).

In [ ]:
!python dataset/distill_rich.py --teacher local --model Qwen/Qwen2.5-7B-Instruct \
  --load_4bit --device cuda --max_tokens 256 --limit 300
# peek at a couple of rich examples
import json
rr = [json.loads(l) for l in open('dataset/plan_dataset_rich.jsonl')]
print('rich rows:', len(rr))
for ex in rr[:2]:
    print('\nINSTR:', ex['instruction']); print('BRIEF:', ex['response_brief']); print('RICH :', ex['response'][:400])

## 2) Retrain the joint planner on the RICH responses
`--data` points at the rich corpus; `max_resp` is larger (rich answers are longer); `lam_kl`
smaller (rich targets already preserve fluency, so less anchoring is needed).

In [ ]:
# 200 train / 50 held / 50 calib = 300 (matches the distilled rows; calib non-empty -> KL active)
!python examples/train_joint.py --data dataset/plan_dataset_rich.jsonl \
  --train 200 --held 50 --calib 50 --epochs 6 --max_resp 96 \
  --accum 8 --lr 1e-4 --lam_kl 0.1 --kl_batch 8 --patience 3 \
  --device cuda --base Qwen/Qwen2.5-0.5B-Instruct

In [ ]:
import pandas as pd, matplotlib.pyplot as plt
df = pd.read_csv('examples/joint_metrics.csv')
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(df.epoch, df.train_total, marker='.', label='train total')
ax[0].plot(df.epoch, df.held_resp_planned, marker='.', label='held response CE (planned)')
ax[0].legend(); ax[0].set_title('losses'); ax[0].set_xlabel('epoch'); ax[0].grid(alpha=.3)
ax[1].plot(df.epoch, df.held_resp_unplanned, marker='.', label='response NLL: NO plan')
ax[1].plot(df.epoch, df.held_resp_planned, marker='.', label='response NLL: WITH plan')
ax[1].legend(); ax[1].set_title('plan ablation (with-plan below = plan helps)')
ax[1].set_xlabel('epoch'); ax[1].grid(alpha=.3)
plt.tight_layout(); plt.show()

## 3) Inference: vanilla Qwen vs planned (now trained on rich targets)

In [ ]:
!python examples/infer_joint.py --ckpt examples/joint_ckpt --max_new 180 \
  --question "I am a novice hiker but I am going to the Himalayas. What are the best hiking equipment for me?"